# Mini Codex：LangChain 版 Codex Harness

这个 Notebook 演示一个**只针对开发场景**的 Codex Harness 最小闭环：

```text
bootstrap → context → plan → agent(model ↔ tools) → verify → finalize
```

它借鉴 Codex 开源实现的核心结构：

- `TurnContext` / `StepContext`：保存本轮上下文；
- `run_turn`：模型返回 tool calls 就执行并继续，返回 assistant message 就结束；
- `EventMsg`：把模型、工具、patch、token、compaction 事件流给 UI；
- `AGENTS.md`：仓库级开发规范进入上下文。

本项目实现：`mini_codex/`  
架构说明：`docs/mini-codex-architecture.md`


In [ ]:
from pathlib import Path
import shutil, tempfile, json
from mini_codex import HarnessConfig, MiniCodexHarness

source = Path("examples/mini_codex_demo").resolve()
workspace = Path(tempfile.mkdtemp(prefix="mini_codex_nb_")) / "demo"
shutil.copytree(source, workspace)
print("示例 workspace:", workspace)


## 1. 查看暴露的工具

Mini Codex 只暴露开发场景需要的工具，不引入 MCP、图片、多 Agent 等额外能力。

In [ ]:
from mini_codex.events import EventBus
from mini_codex.state import ChangeTracker, PlanState
from mini_codex.tools import ToolContext, build_tools

bus = EventBus(workspace / ".mini_codex" / "tool-preview", console=False)
ctx = ToolContext(workspace=workspace, plan=PlanState(), changes=ChangeTracker(), commands=[], event_bus=bus)
tools = build_tools(ctx)
for item in tools:
    print(f"- {item.name}: {item.description}")
bus.close()


## 2. 从头到尾运行一次

下面会在临时复制出来的 demo 仓库中：

1. 读取 `AGENTS.md` 和环境；
2. 生成计划；
3. 让模型读取文件；
4. 修改 `calculator.py`；
5. 运行 pytest；
6. 输出最终总结、diff、事件时间线。

In [ ]:
config = HarnessConfig(
    workspace=workspace,
    max_steps=8,
    token_budget=8000,
    verify_command="python -m pytest -q",
    console=True,
)
harness = MiniCodexHarness(config)
result = harness.run("修复 calculator.py 中 add 函数的 bug，让 pytest 测试通过，并总结修改。")
print("success:", result.success)
print("changed_files:", result.changed_files)
print("run_dir:", result.run_dir)


## 3. 查看可回放产物

每个 run 目录包含：

- `events.jsonl`：逐事件时间线，适合 UI 或机器回放；
- `timeline.md`：阶段耗时；
- `summary.md`：任务、计划、最终回答、命令；
- `changes.diff`：文件差异。

In [ ]:
from IPython.display import Markdown, display
display(Markdown(result.timeline_path.read_text(encoding="utf-8")))
display(Markdown((result.run_dir / "summary.md").read_text(encoding="utf-8")))


In [ ]:
events = [json.loads(line) for line in (result.run_dir / "events.jsonl").read_text(encoding="utf-8").splitlines()]
for event in events:
    stage = event["stage"]
    kind = event["type"]
    message = event["message"].replace("\n", " ")[:120]
    print(f"{stage:10s} {kind:24s} {message}")


## 4. 查看最终 diff

In [ ]:
print((result.run_dir / "changes.diff").read_text(encoding="utf-8"))


## 6. 简略多 Agent 与 MCP

Mini Codex 也支持 `--enable-multi-agent` 和 `--enable-mcp`。

- `spawn_agent` → 子 Agent 返回 `agent_id`
- `send_input` → 写入 mailbox
- `wait_agent` → 读取最终结果
- `repo_stats` / `find_todos` → MCP 动态发现

更完整的通信对照见 `MultiAgentPatterns.ipynb` 和 `docs/multi-agent-communication.md`。


In [ ]:
from mini_codex.multi_agent import build_codex_agent_tools
from mini_codex.events import EventBus

class PreviewModel:
    def bind_tools(self, tools, **kwargs):
        return self
    def invoke(self, messages):
        from langchain_core.messages import AIMessage
        return AIMessage(content="preview")

preview_bus = EventBus(workspace / ".mini_codex" / "multi-preview", console=False)
preview_tools = build_codex_agent_tools(PreviewModel(), [], preview_bus)
print([tool.name for tool in preview_tools])
preview_bus.close()


## 5. 和完整 Codex 的对应关系

| Mini Codex | Codex 开源 Harness |
|---|---|
| `ContextManager` | `TurnContext` / `StepContext` / world state |
| `MiniCodexHarness.run` | `run_turn` 的 sampling/tool loop |
| `EventBus` | `EventMsg` 事件流 |
| `run_command` | `exec_command` / `write_stdin` 的简化版 |
| `write_file` / `replace_in_file` | `apply_patch` 的简化版 |
| `update_plan` | `update_plan` |
| `events.jsonl` / `timeline.md` | 面向 UI 的事件流与可视化 |

完整版 Codex 还包含 OS sandbox、审批 UI、MCP、插件、多 Agent、图片、云端服务等；
本 Notebook 只保留开发 Agent 的必需闭环。